# M2 · Irrigation & alertes météo — entraînement

**Tâche** : prévoir l'évapotranspiration de référence **ET₀** des 7 prochains jours d'une parcelle, en déduire
le besoin en eau de l'olivier (FAO-56) : `besoin = ET₀ × Kc − 0,8 × pluie`, `Kc = 0,65`.

| | |
|---|---|
| Données | Open-Meteo **ERA5** journalier (ET₀ FAO, pluie, Tmax, Tmin), 12 gouvernorats oléicoles, 2014 → aujourd'hui |
| Découpage | **chronologique** : train 2014–2021 · validation 2022 · test 2023 → dernier jour ERA5 |
| Baselines | « même jour l'an dernier » (baseline du plan) et persistance (moyenne des 7 derniers jours) |
| Modèles | **Prophet** (un modèle par région et par date de prévision) et **LSTM** global (toutes régions) |
| Métriques | MAE et RMSE sur l'ET₀ prévue, horizons 1 à 7 jours |
| Export | `models/m2/lstm_et0.npz` + `metadata.json`, lus par `ai-services/m2-irrigation` |

Exécuter de haut en bas sur un noyau frais, depuis ce dossier. Les données brutes sont mises en cache dans
`data/raw/m2_irrigation/` (jamais modifiées), les figures dans `outputs/`.

In [ ]:
import json
import platform

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import m2_common as mc

mc.set_seed()
mc.OUTPUT_DIR.mkdir(exist_ok=True)
RUN_PROPHET = True          # ~10 min sur CPU ; False pour aller vite
PROPHET_EVERY_DAYS = 28     # Prophet est réentraîné à chaque date de prévision : une date tous les 28 jours
pd.set_option("display.precision", 3)
print("python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## 1 · Données Open-Meteo (ERA5)

In [ ]:
raw = mc.load_all()
data, missing = mc.clean(raw)
summary = data.groupby("region").agg(jours=("date", "size"), debut=("date", "min"), fin=("date", "max"),
                                     et0_moy=("et0", "mean"), pluie_an=("rain", lambda r: r.sum() / (len(r) / 365.25)))
print("valeurs manquantes (brutes) :", int(missing.to_numpy().sum()))
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
monthly = data.assign(mois=data["date"].dt.month).groupby(["mois", "region"])["et0"].mean().unstack()
monthly.plot(ax=axes[0], legend=False)
axes[0].set(title="ET₀ moyenne par mois (mm/j) — une courbe par gouvernorat", xlabel="mois", ylabel="mm/j")
yearly_rain = data.assign(an=data["date"].dt.year).groupby(["an", "region"])["rain"].sum().unstack()
yearly_rain[yearly_rain.index < data["date"].max().year].mean(axis=1).plot.bar(ax=axes[1], color="tab:blue")
axes[1].set(title="Pluie annuelle moyenne des 12 gouvernorats (mm)", xlabel="année")
fig.tight_layout()
fig.savefig(mc.OUTPUT_DIR / "eda.png", dpi=110)
plt.show()
data[["et0", "rain", "tmax", "tmin"]].describe()

**Lecture** : l'ET₀ suit un cycle annuel très marqué (≈ 1,5 mm/j en hiver, 6–7 mm/j en juillet), plus fort au sud
(Gabès, Médenine, Sidi Bouzid). La pluie est rare et irrégulière, avec des années sèches (2021–2023).

## 2 · Découpage chronologique et fenêtres

Chaque échantillon = les **30 derniers jours** (ET₀, pluie, Tmax, Tmin, jour de l'année en sin/cos) → l'ET₀ des
**16 jours suivants**. Les fenêtres d'entraînement s'arrêtent 16 jours avant la fin du train : aucune cible
ne déborde sur la validation.

In [ ]:
train_end = pd.Timestamp(mc.TRAIN_END)
val_end = pd.Timestamp(mc.VAL_END)
horizon = pd.Timedelta(days=mc.HORIZON)

norm = mc.Norm.fit(data[data["date"] <= train_end])
et0_mean = float(data.loc[data["date"] <= train_end, "et0"].mean())
et0_std = float(data.loc[data["date"] <= train_end, "et0"].std())

x_train, y_train, i_train = mc.windows(data, norm, mc.DATA_START, train_end - horizon)
x_val, y_val, i_val = mc.windows(data, norm, train_end + pd.Timedelta(days=1), val_end - horizon)
x_test, y_test, i_test = mc.windows(data, norm, val_end + pd.Timedelta(days=1), data["date"].max())
test_start, test_end = i_test[0][1], max(d for _, d in i_test)
print(f"train {x_train.shape} · val {x_val.shape} · test {x_test.shape}")
print(f"test : dates de prévision du {test_start:%Y-%m-%d} au {test_end:%Y-%m-%d}")

## 3 · Baselines

In [ ]:
pred_last_year = mc.baseline_last_year(data, i_test)
pred_persist = mc.baseline_persistence(data, i_test)
pd.DataFrame({
    "même jour l'an dernier": mc.metrics(y_test, pred_last_year),
    "persistance 7 j": mc.metrics(y_test, pred_persist),
}).T

## 4 · LSTM (un modèle pour les 12 régions)

In [ ]:
model, history = mc.train_lstm(x_train, y_train, x_val, y_val, et0_mean, et0_std)
pred_lstm = mc.lstm_predict(model, x_test, et0_mean, et0_std)
best = history.loc[history["val_mae"].idxmin()]
print(f"meilleure époque {int(best['epoch'])} · MAE validation {best['val_mae']:.3f} mm/j")
ax = history.plot(x="epoch", y="val_mae", figsize=(6, 3), title="MAE validation (mm/j)", legend=False)
plt.savefig(mc.OUTPUT_DIR / "lstm_learning_curve.png", dpi=110)
plt.show()
mc.metrics(y_test, pred_lstm)

Le service n'embarque pas PyTorch : il refait le calcul du LSTM en NumPy. On vérifie que les deux donnent le même résultat.

In [ ]:
weights = mc.lstm_weights(model, norm, et0_mean, et0_std)
gap = np.abs(mc.numpy_lstm_forward(weights, x_test[:500].astype(np.float64)) - pred_lstm[:500]).max()
print(f"écart max NumPy vs PyTorch : {gap:.2e} mm/j")
assert gap < 1e-3

## 5 · Prophet

Prophet ne prend qu'une série : un modèle par gouvernorat, **réentraîné à chaque date de prévision** sur les
6 années précédentes (sinon il ne verrait pas les derniers jours). C'est coûteux, donc une date tous les 28 jours.
Tous les modèles sont ensuite comparés sur **ces mêmes dates**.

In [ ]:
subset = [k for k, (_, d) in enumerate(i_test) if (d - test_start).days % PROPHET_EVERY_DAYS == 0]
i_sub = [i_test[k] for k in subset]
y_sub = y_test[subset]
pred_prophet = mc.prophet_forecasts(data, i_sub) if RUN_PROPHET else np.full_like(y_sub, np.nan)
print(f"{len(i_sub)} prévisions Prophet ({len(i_sub) // len(mc.REGIONS)} dates × {len(mc.REGIONS)} régions)")

## 6 · Résultats (jeu de test, jamais vu)

In [ ]:
models_all = {"même jour l'an dernier": pred_last_year, "persistance 7 j": pred_persist, "LSTM": pred_lstm}
table_all = pd.DataFrame({name: mc.metrics(y_test, p) for name, p in models_all.items()}).T
table_all["échantillons"] = len(y_test)

models_sub = {name: p[subset] for name, p in models_all.items()} | {"Prophet": pred_prophet}
table_sub = pd.DataFrame({name: mc.metrics(y_sub, p) for name, p in models_sub.items()}).T
table_sub["échantillons"] = len(y_sub)

results = pd.concat({"toutes les dates de test": table_all, f"dates communes (1 / {PROPHET_EVERY_DAYS} j)": table_sub})
results.to_csv(mc.OUTPUT_DIR / "results.csv")
results

In [ ]:
by_h = pd.DataFrame({name: mc.metrics_by_horizon(y_sub, p) for name, p in models_sub.items()},
                    index=pd.RangeIndex(1, mc.EVAL_HORIZON + 1, name="horizon (jours)"))
ax = by_h.plot(marker="o", figsize=(7, 4), title="MAE de l'ET₀ selon l'horizon (dates communes)")
ax.set_ylabel("MAE (mm/j)")
plt.savefig(mc.OUTPUT_DIR / "mae_by_horizon.png", dpi=110)
plt.show()
by_h

**Ce que l'erreur représente pour l'agriculteur** : le besoin en eau vaut `ET₀ × 0,65`, donc une erreur de
1 mm/j sur l'ET₀ = 0,65 mm/j d'eau, soit **0,65 L par m²**. Pour une parcelle type de 2,5 ha et 250 oliviers
(100 m² par arbre) :

In [ ]:
m2_per_tree = 2.5 * 10_000 / 250
liters = (table_all["MAE"] * 0.65 * m2_per_tree).rename("erreur moyenne (L / arbre / jour)")
liters.to_frame()

In [ ]:
region, start = "sfax", pd.Timestamp(f"{test_start.year + 1}-06-01")
k = next(i for i, (r, d) in enumerate(i_test) if r == region and d == start)
days = pd.date_range(start + pd.Timedelta(days=1), periods=mc.EVAL_HORIZON)
plt.figure(figsize=(7, 3.5))
plt.plot(days, y_test[k, :7], "k-o", label="observé (ERA5)")
for name, p in [("LSTM", pred_lstm), ("même jour l'an dernier", pred_last_year), ("persistance 7 j", pred_persist)]:
    plt.plot(days, p[k, :7], "--", label=name)
plt.title(f"Exemple : ET₀ prévue à {region.capitalize()} depuis le {start:%d/%m/%Y}")
plt.ylabel("mm/j")
plt.legend()
plt.savefig(mc.OUTPUT_DIR / "example_sfax.png", dpi=110)
plt.show()

## 7 · Choix du modèle et export

On sert le modèle qui a la plus petite MAE sur le test. Dans le service, l'ET₀ vient du modèle ; la pluie et les
températures des jours à venir (alertes gel / canicule) viennent des **prévisions Open-Meteo** : aucun modèle
statistique ne prévoit la pluie mieux qu'un modèle météo numérique.

In [ ]:
mae_sub = table_sub["MAE"]
print("meilleur modèle sur les dates communes :", mae_sub.idxmin())
if mae_sub.idxmin() != "LSTM":
    print("attention : le LSTM n'est pas le meilleur sur ce test, à signaler dans le rapport")
gain = 1 - table_all.loc["LSTM", "MAE"] / table_all.loc["même jour l'an dernier", "MAE"]
print(f"LSTM vs baseline 'même jour l'an dernier' : MAE -{gain:.0%}")

import torch

metadata = {
    "model": "LSTM global (1 couche, 64 unités) -> ET0 J+1..J+16",
    "data": {
        "source": "Open-Meteo ERA5 (archive-api.open-meteo.com)",
        "regions": list(mc.REGIONS),
        "period": [str(data["date"].min().date()), str(data["date"].max().date())],
        "split": {"train": [mc.DATA_START, mc.TRAIN_END], "val": ["2022-01-01", mc.VAL_END],
                  "test": [str(test_start.date()), str(test_end.date())]},
    },
    "metrics_test_all_origins": json.loads(table_all.to_json(orient="index")),
    "metrics_test_common_origins": json.loads(table_sub.to_json(orient="index")),
    "best_on_common_origins": mae_sub.idxmin(),
    "versions": {"python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__},
}
mc.export(weights, metadata)